# Démonstration de certification qualité AssurConnect

**Durée cible : 2 minutes.** Cette démonstration interroge en lecture seule le certificat réel et les mesures de règles bloquantes du lot `L-2026-10-14`. Elle n’écrit pas dans la base et ne simule aucun résultat.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
from sqlalchemy import text

candidates = [Path.cwd(), Path.cwd().parent, Path("/home/jovyan/work")]
repo_root = next((path for path in candidates if (path / "pipelines" / "certify_s5.py").is_file()), None)
if repo_root is None:
    raise FileNotFoundError("Le dépôt AssurConnect et pipelines/certify_s5.py sont introuvables.")
sys.path.insert(0, str(repo_root / "pipelines"))
import certify_s5

batch_id = "L-2026-10-14"
certificate_id = "CERT-L20261014-132851"
layer_by_gate = {"B": "BRONZE", "S": "SILVER", "G": "GOLD", "C": "CRM"}
engine = certify_s5.create_engine(certify_s5.DEFAULT_URL)

with engine.connect() as connection:
    certificate = connection.execute(
        text("""SELECT certificat_id, batch_id, nb_bloquantes_testees,
                      nb_bloquantes_ok, empreinte_gold, decision, signataire, emis_le
               FROM dq.certificat_qualite WHERE certificat_id=:certificate_id"""),
        {"certificate_id": certificate_id},
    ).mappings().first()
    if certificate is None:
        raise LookupError(f"Certificat introuvable : {certificate_id}")

    rules = connection.execute(
        text("SELECT rule_id, porte FROM dq.regle WHERE severite='BLOQUANTE' ORDER BY rule_id")
    ).mappings().all()
    history = connection.execute(
        text("""SELECT h.rule_id, h.couche, h.nb_testees, h.nb_echecs
               FROM dq.dq_history h
               JOIN (SELECT rule_id, couche, table_cible, MAX(id) AS max_id
                     FROM dq.dq_history WHERE batch_id=:batch_id
                     GROUP BY rule_id, couche, table_cible) m ON m.max_id=h.id
               WHERE h.batch_id=:batch_id"""),
        {"batch_id": batch_id},
    ).mappings().all()

measurements = {}
for row in history:
    key = (row["rule_id"], row["couche"])
    totals = measurements.setdefault(key, {"tested": 0, "failures": 0})
    totals["tested"] += int(row["nb_testees"])
    totals["failures"] += int(row["nb_echecs"])

coverage = []
for rule in rules:
    expected_layer = layer_by_gate[rule["porte"]]
    metric = measurements.get((rule["rule_id"], expected_layer))
    measured = bool(metric and metric["tested"] > 0)
    passed = bool(measured and metric["failures"] == 0)
    coverage.append({
        "Règle": rule["rule_id"],
        "Couche attendue": expected_layer,
        "Tests": metric["tested"] if metric else 0,
        "Échecs": metric["failures"] if metric else "—",
        "Mesurée": "Oui" if measured else "Non",
        "Résultat": "Conforme" if passed else ("Échec" if measured else "Non mesurée"),
    })

covered = sum(row["Mesurée"] == "Oui" for row in coverage)
passed = sum(row["Résultat"] == "Conforme" for row in coverage)
if covered != int(certificate["nb_bloquantes_testees"]) or passed != int(certificate["nb_bloquantes_ok"]):
    raise ValueError("Les mesures recalculées ne correspondent pas au certificat enregistré.")

print(f"CERTIFICAT {certificate['certificat_id']} | LOT {certificate['batch_id']} | DÉCISION {certificate['decision']}")
print(f"Couverture bloquante : {covered}/{len(coverage)} mesurées dans la couche attendue; {passed} conformes")
print(f"Signataire : {certificate['signataire'] or 'aucun'} | Émis le : {certificate['emis_le']}")
print(f"SHA-256 Gold : {certificate['empreinte_gold']}")
display(pd.DataFrame(coverage))

CERTIFICAT CERT-L20261014-132851 | LOT L-2026-10-14 | DÉCISION NO_GO
Couverture bloquante : 2/19 mesurées dans la couche attendue; 2 conformes
Signataire : aucun | Émis le : 2026-10-21 13:28:51
SHA-256 Gold : 85b9884af4b72ecb46a8784e86402625befbc5e6eb41cc1091fbd6e9ae05e01e


,Règle,Couche attendue,Tests,Échecs,Mesurée,Résultat
0,RQ-001,SILVER,0,—,Non,Non mesurée
1,RQ-002,GOLD,0,—,Non,Non mesurée
2,RQ-007,SILVER,0,—,Non,Non mesurée
3,RQ-009,SILVER,0,—,Non,Non mesurée
4,RQ-010,SILVER,0,—,Non,Non mesurée
5,RQ-011,GOLD,0,—,Non,Non mesurée
6,RQ-012,GOLD,0,—,Non,Non mesurée
7,RQ-013,GOLD,0,—,Non,Non mesurée
8,RQ-014,SILVER,0,—,Non,Non mesurée
9,RQ-015,GOLD,0,—,Non,Non mesurée
